[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_49_Eval_at_Scale.ipynb)

# Lesson 49 — Eval at Scale: Continuous Quality Monitoring in Production

**Track 5 Progress: L46 ✅ Durable Execution → L47 ✅ GPU Autoscaling → L48 ✅ OTel Tracing → **L49 ← YOU ARE HERE** → L50 Capstone**

---

## The core problem: CI eval ≠ production eval

In lessons L17 and L24–L31 you built eval harnesses. Those run on a **golden set in CI** — 50–200 queries, fast, cheap, gating deployments. But once your agent is live:

| Dimension | CI Eval | Production Eval |
|-----------|---------|----------------|
| Volume | 50–200 queries | 10K–1M queries/day |
| Latency budget | Minutes | Async, hours OK |
| Cost budget | Run everything | Must sample |
| Data | Golden set | Real user traffic |
| Goal | Gate deploys | Detect drift early |
| Frequency | On commit | Continuous |

You can't run an LLM judge on every production request:
- 100K requests/day × $0.001/judge call = **$3,000/month** just on eval
- Judging adds latency to the hot path
- You need trends over time, not per-request verdicts

This lesson teaches you to eval at production scale: **sample smartly, judge cheaply, detect drift statistically, alert early**.

### What you'll build
1. **EvalSampler** — 4 strategies (random, stratified, importance, reservoir)
2. **AsyncEvalPipeline** — decouple eval from inference via asyncio.Queue
3. **BatchJudge** — Anthropic Message Batches API (50% cheaper)
4. **EvalMetricStore** — SQLite-backed time-series quality storage
5. **RegressionDetector** — Mann-Whitney U + z-test for drift detection
6. **OTel integration** — eval metrics as Prometheus-ready instruments
7. **ProductionEvalSystem** — full wiring into one deployable class

## Architecture: how eval at scale works

```
Production Traffic
     │
     ▼
┌──────────────┐
│  Inference   │  ← hot path, latency-sensitive
│  Pipeline    │
└──────┬───────┘
       │ sample 1-5%
       ▼
┌──────────────┐
│  Eval Queue  │  ← async, decoupled from hot path
│  (asyncio.Q) │    ~1μs enqueue, never blocks user
└──────┬───────┘
       │ batch up to N
       ▼
┌──────────────┐     ┌──────────────┐
│ Batch Judge  │────▶│ Metric Store │
│(Batches API) │     │  (SQLite/PG) │
│ 50% cheaper  │     │ time-series  │
└──────────────┘     └──────┬───────┘
                            │ aggregate
                            ▼
                    ┌──────────────┐
                    │ Regression   │
                    │  Detector    │
                    │Mann-Whitney U│
                    └──────┬───────┘
                           │ alert
                           ▼
                    ┌──────────────┐
                    │  OTel Meter  │
                    │  Dashboard   │
                    │  + Alerts    │
                    └──────────────┘
```

**Three design rules:**
1. **Decouple eval from inference** — never block a user request to run a judge
2. **Sample, don't judge everything** — 1–5% of traffic, stratified by risk
3. **Batch judge calls** — Anthropic Message Batches API = 50% cost reduction

These rules work together: if you decouple first, you can afford to be patient and batch calls. If you sample smartly, you keep cost down while maintaining signal quality.

In [ ]:
# Cell 3: Setup — install dependencies
!pip install anthropic nest_asyncio pandas matplotlib scipy opentelemetry-api opentelemetry-sdk --quiet

import anthropic
import asyncio
import time
import uuid
import json
import sqlite3
import random
from dataclasses import dataclass, field
from typing import Optional, Literal
from collections import Counter
from datetime import datetime, timedelta
from enum import Enum

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
import nest_asyncio
nest_asyncio.apply()

# OTel metrics
from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import (
    ConsoleMetricExporter,
    PeriodicExportingMetricReader,
)

# ── Constants ───────────────────────────────────────────────────────────────
HAIKU  = "claude-haiku-4-5"
SONNET = "claude-sonnet-4-6"
client = anthropic.Anthropic()

random.seed(42)  # reproducible demos
print("✅ Setup complete")

## Part 1: Sampling strategies

You can't eval everything. The goal: sample traffic that is **representative** AND **high-signal**.

| Strategy | When to use | How |
|----------|-------------|-----|
| **Random** | Simple baseline | `random.random() < rate` |
| **Stratified** | Different query types have different risk | Higher rate for risky tags |
| **Importance** | Focus on edge cases | Higher rate for low-confidence, long, expensive |
| **Reservoir** | Fixed-size budget over unbounded stream | Knuth algorithm, uniform coverage |

**The golden rule:** sample at a rate where you get at least **100 judged samples per day** per metric you care about. With 100 samples, your 95% CI on a proportion is roughly ±10 percentage points — narrow enough to detect real regressions.

**Practical defaults:**
- Base rate: 2–5% for high-volume systems (>10K req/day)
- Code generation: 3× base rate (harder to judge, more likely to fail)
- Always 100% sample: errors, refusals, requests with `latency > SLO`

In [ ]:
# Cell 5: EvalSampler — 4 strategies

class SampleStrategy(str, Enum):
    RANDOM     = "random"
    STRATIFIED = "stratified"
    IMPORTANCE = "importance"
    RESERVOIR  = "reservoir"


@dataclass
class EvalSample:
    """One production request + its response, ready for eval judging."""
    id:           str
    query:        str
    response:     str
    model:        str
    latency_s:    float
    cost_usd:     float
    tag:          str           # "search" | "summarize" | "qa" | "code" | etc.
    confidence:   float         # 0–1, from model verbalized confidence or heuristic
    ts:           datetime = field(default_factory=datetime.utcnow)
    judge_score:  Optional[float] = None
    judge_verdict:Optional[str]   = None


class EvalSampler:
    """Decides which production requests to route for eval judging."""

    def __init__(
        self,
        strategy:       SampleStrategy = SampleStrategy.RANDOM,
        base_rate:      float          = 0.05,
        tag_rates:      dict           = None,   # per-tag override for STRATIFIED
        reservoir_size: int            = 1000,
    ):
        self.strategy       = strategy
        self.base_rate      = base_rate
        self.tag_rates      = tag_rates or {}
        self._reservoir:    list = []
        self._reservoir_size= reservoir_size
        self._seen          = 0

    def should_sample(self, sample: EvalSample) -> bool:
        """Return True if this request should be sent for eval."""
        # Always sample errors
        if sample.judge_verdict == "error" or sample.latency_s > 10:
            return True

        if self.strategy == SampleStrategy.RANDOM:
            return random.random() < self.base_rate

        elif self.strategy == SampleStrategy.STRATIFIED:
            rate = self.tag_rates.get(sample.tag, self.base_rate)
            return random.random() < rate

        elif self.strategy == SampleStrategy.IMPORTANCE:
            # Importance score: higher for low confidence, long responses, high cost
            score = (
                (1 - sample.confidence)                       * 0.50 +  # low conf → sample more
                min(len(sample.response) / 2000, 1.0)         * 0.30 +  # long response
                min(sample.cost_usd / 0.01, 1.0)              * 0.20    # expensive call
            )
            # Blend base_rate with importance score
            effective_rate = self.base_rate + score * (min(0.50, self.base_rate * 5) - self.base_rate)
            return random.random() < effective_rate

        elif self.strategy == SampleStrategy.RESERVOIR:
            # Knuth reservoir: every item has equal probability of being in reservoir
            self._seen += 1
            if len(self._reservoir) < self._reservoir_size:
                self._reservoir.append(sample)
                return True
            j = random.randint(0, self._seen - 1)
            if j < self._reservoir_size:
                self._reservoir[j] = sample
                return True
            return False

        return False


# ── Helper: make a fake production sample ───────────────────────────────────
def make_fake_sample(tag: str, confidence: float = None) -> EvalSample:
    return EvalSample(
        id=str(uuid.uuid4())[:8],
        query=f"Sample {tag} query",
        response="Content. " * random.randint(10, 200),
        model=HAIKU,
        latency_s=random.uniform(0.2, 3.0),
        cost_usd=random.uniform(0.0005, 0.005),
        tag=tag,
        confidence=confidence if confidence is not None else random.random(),
    )


# ── Demo: compare sampling distributions on 10,000 requests ─────────────────
N = 10_000
requests_by_tag = {"qa": 4000, "summarize": 3000, "search": 2000, "code": 1000}

samplers = {
    "Random 5%":  EvalSampler(SampleStrategy.RANDOM, base_rate=0.05),
    "Stratified": EvalSampler(SampleStrategy.STRATIFIED, base_rate=0.02,
                               tag_rates={"code": 0.20, "qa": 0.03,
                                          "summarize": 0.02, "search": 0.01}),
    "Importance": EvalSampler(SampleStrategy.IMPORTANCE, base_rate=0.02),
}

results = {name: Counter() for name in samplers}
for tag, count in requests_by_tag.items():
    for _ in range(count):
        conf = 0.50 if tag == "code" else 0.88  # code has lower confidence
        s = make_fake_sample(tag, confidence=conf)
        for name, sampler in samplers.items():
            if sampler.should_sample(s):
                results[name][tag] += 1

df = pd.DataFrame(results).T
df["total"] = df.sum(axis=1)
df["rate%"]  = (df["total"] / N * 100).round(1)

print(f"Sampling comparison on {N:,} requests:\n")
print(df.to_string())
print()
print("Key insight: Importance sampling over-selects 'code' (low confidence)")
print("vs Random which is proportional to volume. Stratified gives explicit control.")

## Part 2: Async eval pipeline — decouple from the hot path

The most important architectural decision: **eval must never add latency to the user response**.

```
Request arrives
    │
    ├─ Inference → response to user  (synchronous, latency-sensitive)
    │
    └─ if sampled: queue.put_nowait()  (~1μs, non-blocking)

Background worker (separate task, always running):
    └─ pop batch from queue
       → judge batch (async, slow is OK)
       → write scores to MetricStore
```

The queue is bounded (`maxsize`). If it fills up, we **drop** the eval sample — never block the user. Dropping eval samples slightly reduces measurement precision but doesn't impact users at all.

In production you'd use Redis or a message queue (SQS, Pub/Sub) so multiple inference pods can share one eval worker. For now, `asyncio.Queue` demonstrates the same pattern.

In [ ]:
# Cell 7: AsyncEvalPipeline — queue-based decoupling

@dataclass
class JudgeRequest:
    sample:       EvalSample
    submitted_at: datetime = field(default_factory=datetime.utcnow)


class AsyncEvalPipeline:
    """
    Non-blocking eval pipeline. Inference calls `maybe_enqueue()` (1μs).
    A background worker drains the queue, judges, and writes metrics.
    """

    def __init__(
        self,
        sampler:        EvalSampler,
        batch_size:     int = 10,
        max_queue_size: int = 500,
    ):
        self.sampler      = sampler
        self.batch_size   = batch_size
        self._queue: asyncio.Queue = asyncio.Queue(maxsize=max_queue_size)
        self._judged:  list = []
        self._enqueued: int = 0
        self._dropped:  int = 0

    def maybe_enqueue(self, sample: EvalSample) -> bool:
        """
        Call after every production inference. ~1μs. Never blocks.
        Returns True if sample was accepted for eval.
        """
        if not self.sampler.should_sample(sample):
            return False
        try:
            self._queue.put_nowait(JudgeRequest(sample=sample))
            self._enqueued += 1
            return True
        except asyncio.QueueFull:
            # Queue full → drop gracefully. NEVER block.
            self._dropped += 1
            return False

    async def _judge_one(self, s: EvalSample) -> EvalSample:
        """Inline judge for Colab. In production use BatchJudge (Part 3)."""
        prompt = (
            f"Rate this AI response quality (0.0–1.0).\n"
            f"Query: {s.query[:200]}\n"
            f"Response: {s.response[:400]}\n\n"
            f'Return JSON only: {{"score": <0.0-1.0>, "verdict": "<good|ok|poor>", "reason": "<10 words>"}}'
        )
        try:
            msg = client.messages.create(
                model=HAIKU,
                max_tokens=80,
                messages=[{"role": "user", "content": prompt}],
            )
            raw = msg.content[0].text.strip().replace("```json", "").replace("```", "").strip()
            data = json.loads(raw)
            s.judge_score   = float(data.get("score",   0.5))
            s.judge_verdict = data.get("verdict", "ok")
        except Exception:
            s.judge_score   = 0.5
            s.judge_verdict = "ok"
        return s

    async def _judge_batch(self, batch: list[EvalSample]) -> list[EvalSample]:
        # For Colab: judge concurrently within the batch
        return await asyncio.gather(*[self._judge_one(s) for s in batch])

    async def _worker(self, max_batches: int = 5):
        """Background worker: drain queue in batches and judge."""
        batches_done = 0
        while batches_done < max_batches:
            batch = []
            # Non-blocking drain up to batch_size
            while len(batch) < self.batch_size:
                try:
                    item = self._queue.get_nowait()
                    batch.append(item.sample)
                except asyncio.QueueEmpty:
                    break

            if not batch:
                await asyncio.sleep(0.05)
                continue

            judged = await self._judge_batch(batch)
            self._judged.extend(judged)
            batches_done += 1
            scores = [s.judge_score for s in judged]
            print(f"  Batch {batches_done}: judged {len(batch)} samples | "
                  f"mean_score={sum(scores)/len(scores):.3f}")

    async def run_demo(self, n_requests: int = 60):
        """Simulate n_requests through the pipeline (enqueue → worker)."""
        topics = ["transformers", "RAG", "LoRA", "vLLM", "OTel", "DPO", "RLHF"]
        for i in range(n_requests):
            tag = random.choice(["qa", "summarize", "search", "code"])
            s   = make_fake_sample(tag)
            s.query    = f"Query #{i}: explain {random.choice(topics)}"
            s.response = f"Response #{i}. " + "Detail. " * random.randint(5, 40)
            self.maybe_enqueue(s)

        print(f"  Inference done: {n_requests} requests processed")
        print(f"  Enqueued for eval: {self._enqueued} | Dropped (queue full): {self._dropped}")
        print(f"  Queue depth now: {self._queue.qsize()}")
        print()
        print("Running background eval worker...")
        await self._worker(max_batches=4)
        return self._judged


# ── Run demo ─────────────────────────────────────────────────────────────────
pipeline = AsyncEvalPipeline(
    sampler    = EvalSampler(SampleStrategy.IMPORTANCE, base_rate=0.20),
    batch_size = 6,
)
judged = asyncio.run(pipeline.run_demo(n_requests=70))

print(f"\n✅ Total judged: {len(judged)}")
print(f"   Mean score : {sum(s.judge_score for s in judged)/len(judged):.3f}")
print(f"   Verdicts   : {Counter(s.judge_verdict for s in judged)}")

## Part 3: Batch LLM judging — 50% cost reduction with Message Batches API

Anthropic's **Message Batches API** lets you submit up to 10,000 judge calls in one request and poll for results asynchronously. Cost is **50% of real-time** API pricing. Perfect for overnight eval runs.

```
submit_batch(requests=[...])  →  batch_id
    │
    ├─ poll every 30s: retrieve(batch_id).processing_status
    │
    └─ when status == "ended":
         iterate batches.results(batch_id)  →  custom_id + result
```

Key differences from real-time:
- No streaming support
- Results are not ordered — use `custom_id` to map back
- SLA is "within 24 hours" (usually minutes)
- Best for: **nightly eval runs, bulk annotation, CI overnight jobs**

```python
# Minimal batch pattern
batch = client.messages.batches.create(requests=[{
    "custom_id": sample.id,
    "params": {"model": HAIKU, "max_tokens": 100,
               "messages": [{"role": "user", "content": judge_prompt(sample)}]}
} for sample in samples])

# Poll
while client.messages.batches.retrieve(batch.id).processing_status != "ended":
    time.sleep(30)

# Collect results
for result in client.messages.batches.results(batch.id):
    sample = id_to_sample[result.custom_id]
    sample.judge_score = parse_score(result.result.message)
```

In [ ]:
# Cell 9: BatchJudge — Anthropic Message Batches API

class BatchJudge:
    """
    Submits a batch of eval samples to Anthropic Message Batches API.
    50% cheaper than real-time; best for overnight eval runs.
    """

    # Tool-forced JSON for reliable score extraction
    JUDGE_TOOL = {
        "name": "submit_quality_score",
        "description": "Submit quality evaluation for an AI-generated response",
        "input_schema": {
            "type": "object",
            "properties": {
                "score":         {"type": "number",
                                  "description": "Quality score 0.0 (terrible) to 1.0 (excellent)"},
                "verdict":       {"type": "string", "enum": ["good", "ok", "poor"]},
                "primary_issue": {"type": "string",
                                  "description": "Main issue if not good, else empty string"},
            },
            "required": ["score", "verdict"],
        }
    }

    def __init__(self, model: str = HAIKU, max_tokens: int = 150):
        self.model      = model
        self.max_tokens = max_tokens

    def _judge_prompt(self, s: EvalSample) -> str:
        return (
            f"You are an AI quality evaluator. Rate the response below.\n\n"
            f"Query: {s.query[:300]}\n\n"
            f"Response: {s.response[:600]}\n\n"
            f"Use the submit_quality_score tool to return your evaluation."
        )

    def _build_requests(self, samples: list[EvalSample]) -> list[dict]:
        return [
            {
                "custom_id": s.id,
                "params": {
                    "model":       self.model,
                    "max_tokens":  self.max_tokens,
                    "tools":       [self.JUDGE_TOOL],
                    "tool_choice": {"type": "tool", "name": "submit_quality_score"},
                    "messages":    [{"role": "user", "content": self._judge_prompt(s)}],
                }
            }
            for s in samples
        ]

    def submit_batch(self, samples: list[EvalSample]) -> str:
        """Submit samples for batch judging. Returns batch_id."""
        requests = self._build_requests(samples)
        batch    = client.messages.batches.create(requests=requests)
        print(f"  Batch submitted  : {batch.id}")
        print(f"  Requests in batch: {len(requests)}")
        print(f"  Initial status   : {batch.processing_status}")
        return batch.id

    def poll_until_done(self, batch_id: str, poll_s: int = 5, timeout_s: int = 120) -> bool:
        """Poll until batch is done. Returns True on success."""
        start = time.time()
        while True:
            batch  = client.messages.batches.retrieve(batch_id)
            counts = batch.request_counts
            print(f"  [{batch.processing_status}] "
                  f"processing={counts.processing} "
                  f"succeeded={counts.succeeded} "
                  f"errored={counts.errored}")
            if batch.processing_status == "ended":
                return True
            if time.time() - start > timeout_s:
                return False
            time.sleep(poll_s)

    def collect_results(
        self,
        batch_id:    str,
        id_to_sample: dict[str, EvalSample],
    ) -> list[EvalSample]:
        """Parse batch results and attach judge scores to samples."""
        scored = []
        for result in client.messages.batches.results(batch_id):
            s = id_to_sample.get(result.custom_id)
            if s is None:
                continue
            if result.result.type == "succeeded":
                for block in result.result.message.content:
                    if block.type == "tool_use" and block.name == "submit_quality_score":
                        s.judge_score   = float(block.input.get("score",   0.5))
                        s.judge_verdict = block.input.get("verdict", "ok")
                        break
            else:
                # errored / expired → default
                s.judge_score   = 0.5
                s.judge_verdict = "ok"
            scored.append(s)
        return scored


# ── Demo: submit a real batch of 6 samples ───────────────────────────────────
print("=== BatchJudge Demo ===\n")

GOOD_RESPONSE = (
    "Transformers use multi-head self-attention to compute pairwise "
    "relationships between all tokens in a sequence. Each attention "
    "head learns to focus on different types of relationships."
)
OK_RESPONSE = "Transformers pay attention to words."
POOR_RESPONSE = "Transformers are robots that transform. They are popular in movies."

batch_samples = []
for response, tag in [
    (GOOD_RESPONSE, "qa"),
    (OK_RESPONSE,   "qa"),
    (POOR_RESPONSE, "qa"),
    (GOOD_RESPONSE, "summarize"),
    (OK_RESPONSE,   "code"),
    (POOR_RESPONSE, "code"),
]:
    s = make_fake_sample(tag)
    s.query    = "Explain how transformers work in AI"
    s.response = response
    batch_samples.append(s)

judge        = BatchJudge(model=HAIKU)
id_to_sample = {s.id: s for s in batch_samples}

print("Submitting batch...")
batch_id = judge.submit_batch(batch_samples)

print("\nPolling for completion...")
ok = judge.poll_until_done(batch_id, poll_s=3, timeout_s=90)

if ok:
    scored = judge.collect_results(batch_id, id_to_sample)
    print(f"\n✅ Batch complete. Scored {len(scored)} samples.\n")
    for s in scored:
        print(f"  [{s.judge_verdict:4s}] score={s.judge_score:.2f} | "
              f"response='{s.response[:55]}...'")
    n   = len(batch_samples)
    rt  = n * 0.000125  # typical Haiku cost per call
    bat = rt * 0.50
    print(f"\n💰 Cost estimate for {n} judge calls:")
    print(f"   Real-time: ${rt:.4f}")
    print(f"   Batch API: ${bat:.4f}  (50% off)")
    print(f"   At 10K calls/day: ${rt*10000:.2f}/day real-time vs ${bat*10000:.2f}/day batch")
else:
    print("⚠️  Batch timed out — using fallback scores for rest of notebook")
    for s in batch_samples:
        s.judge_score   = random.uniform(0.4, 0.95)
        s.judge_verdict = random.choice(["good", "ok", "poor"])
    scored = batch_samples

## Part 4: Time-series metric storage

Eval scores are only useful as **trends over time**. A single score means nothing. A score that drops 10% over 48 hours is a signal.

You need to store scores with:
- `ts` — timestamp for trend queries
- `tag` — to distinguish QA drift from code drift  
- `model` — to attribute changes to model updates
- Raw sample IDs — so you can investigate individual failures

**Schema rule:** store **raw samples**, not just aggregates. Aggregates are cheap to recompute; raw samples let you debug regressions by looking at the actual failing queries.

**Stack recommendation:**
| Scale | Storage |
|-------|--------|
| < 1M records | SQLite (zero ops overhead) |
| 1M–100M records | PostgreSQL + TimescaleDB extension |
| > 100M records | ClickHouse (columnar, fast aggregations) |

In [ ]:
# Cell 11: EvalMetricStore — SQLite-backed time-series storage

class EvalMetricStore:
    """
    Stores eval scores with timestamps for trend analysis.
    SQLite for dev; swap db_path for PostgreSQL URI in prod.
    """

    _SCHEMA = """
        CREATE TABLE IF NOT EXISTS eval_scores (
            id           TEXT PRIMARY KEY,
            ts           TEXT NOT NULL,
            model        TEXT NOT NULL,
            tag          TEXT NOT NULL,
            judge_score  REAL NOT NULL,
            judge_verdict TEXT NOT NULL,
            latency_s    REAL,
            cost_usd     REAL,
            query_len    INTEGER,
            response_len INTEGER
        );
        CREATE INDEX IF NOT EXISTS idx_ts  ON eval_scores(ts);
        CREATE INDEX IF NOT EXISTS idx_tag ON eval_scores(tag, ts);
    """

    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        for stmt in self._SCHEMA.strip().split(";"):
            if stmt.strip():
                self.conn.execute(stmt)
        self.conn.commit()

    def write(self, samples: list[EvalSample]) -> int:
        """Write scored samples. Returns number of rows written."""
        rows = [
            (
                s.id, s.ts.isoformat(), s.model, s.tag,
                s.judge_score, s.judge_verdict or "ok",
                s.latency_s, s.cost_usd,
                len(s.query), len(s.response),
            )
            for s in samples if s.judge_score is not None
        ]
        self.conn.executemany(
            "INSERT OR REPLACE INTO eval_scores VALUES (?,?,?,?,?,?,?,?,?,?)", rows
        )
        self.conn.commit()
        return len(rows)

    def query_window(self, start: datetime, end: datetime, tag: str = None) -> pd.DataFrame:
        """Return all scores in [start, end], optionally filtered by tag."""
        sql    = "SELECT * FROM eval_scores WHERE ts >= ? AND ts <= ?"
        params = [start.isoformat(), end.isoformat()]
        if tag:
            sql    += " AND tag = ?"
            params.append(tag)
        return pd.read_sql_query(sql, self.conn, params=params, parse_dates=["ts"])

    def rolling_quality(self) -> pd.DataFrame:
        """Hourly quality aggregation per tag — for dashboard and drift detection."""
        sql = """
        SELECT
            strftime('%Y-%m-%dT%H:00:00', ts) AS hour,
            tag,
            COUNT(*)                           AS n,
            AVG(judge_score)                   AS mean_score,
            SUM(CASE WHEN judge_verdict = 'poor'
                     THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS poor_rate
        FROM eval_scores
        GROUP BY hour, tag
        ORDER BY hour
        """
        return pd.read_sql_query(sql, self.conn)


# ── Populate with 7 days of synthetic history ────────────────────────────────
store    = EvalMetricStore()
base_time = datetime.utcnow() - timedelta(days=7)

# Quality baseline (days 0–4) and degradation (days 5–6)
BASE_SCORES = {"qa": 0.82, "summarize": 0.78, "search": 0.85, "code": 0.70}

all_samples: list[EvalSample] = []
for day in range(7):
    drift = -0.18 if day >= 5 else 0.0   # simulated model degradation after day 5
    for hour in range(0, 24, 2):
        for tag in ["qa", "summarize", "search", "code"]:
            for _ in range(random.randint(4, 9)):
                s = make_fake_sample(tag)
                s.ts = base_time + timedelta(
                    days=day, hours=hour, minutes=random.randint(0, 59)
                )
                raw_score   = BASE_SCORES[tag] + drift + random.gauss(0, 0.08)
                s.judge_score   = max(0.0, min(1.0, raw_score))
                s.judge_verdict = (
                    "good" if s.judge_score > 0.75
                    else ("ok" if s.judge_score > 0.50 else "poor")
                )
                all_samples.append(s)

written = store.write(all_samples)
print(f"✅ Wrote {written:,} eval scores to metric store (7 days of history)")
print(f"   Tags: {list(BASE_SCORES.keys())}")
print(f"   Quality drift injected on days 5–6 (−0.18 offset)\n")

df_q = store.rolling_quality()
print(f"Rolling quality sample (last 5 rows):")
print(df_q.tail(5).to_string(index=False))

## Part 5: Regression detection — statistical tests, not thresholds

A fixed threshold like "alert if mean_score < 0.7" has two problems:
1. **False positives** on low-volume windows (3 bad samples in 1 hour → alert)
2. **Misses gradual drift** — score moves from 0.82 → 0.72 over 2 days, never crossing a static threshold

Better approach: **compare two time windows statistically**.

```
Baseline window: last 48h before the current window
Current window : last 6h

Test 1: Mann-Whitney U
  → Does the score DISTRIBUTION shift between windows?
  → Non-parametric (doesn't assume normality)
  → alternative='less' tests if current < baseline

Test 2: Two-proportion z-test on poor_rate
  → Did the fraction of 'poor' verdicts increase?
  → More sensitive than mean for quality regressions
```

Both tests require **min_samples** before firing. Don't alert on 5-sample windows.

In [ ]:
# Cell 13: RegressionDetector — Mann-Whitney U + z-test

@dataclass
class RegressionAlert:
    kind:           Literal["quality_drop", "poor_rate_spike"]
    tag:            str
    baseline_mean:  float
    current_mean:   float
    delta:          float
    p_value:        float
    severity:       Literal["warning", "critical"]
    message:        str


class RegressionDetector:
    """
    Compares current eval window against baseline using:
    - Mann-Whitney U test (score distribution shift)
    - Two-proportion z-test (poor rate increase)
    """

    def __init__(
        self,
        baseline_hours:  int   = 48,
        current_hours:   int   = 6,
        min_samples:     int   = 30,
        p_threshold:     float = 0.05,
        delta_warning:   float = 0.05,   # 5 pp drop → warning
        delta_critical:  float = 0.10,   # 10 pp drop → critical
    ):
        self.baseline_hours = baseline_hours
        self.current_hours  = current_hours
        self.min_samples    = min_samples
        self.p_threshold    = p_threshold
        self.delta_warning  = delta_warning
        self.delta_critical = delta_critical

    def detect(self, store: EvalMetricStore, now: datetime = None) -> list[RegressionAlert]:
        now  = now or datetime.utcnow()
        alerts: list[RegressionAlert] = []

        cur_start  = now - timedelta(hours=self.current_hours)
        base_start = now - timedelta(hours=self.baseline_hours)

        df_cur  = store.query_window(cur_start,  now)
        df_base = store.query_window(base_start, cur_start)

        if df_cur.empty or df_base.empty:
            return alerts

        for tag in list(df_cur["tag"].unique()) + ["__all__"]:
            label = "overall" if tag == "__all__" else tag
            if tag == "__all__":
                cur_scores  = df_cur["judge_score"].values
                base_scores = df_base["judge_score"].values
                poor_cur    = (df_cur["judge_verdict"]  == "poor").mean()
                poor_base   = (df_base["judge_verdict"] == "poor").mean()
                n_cur       = len(cur_scores)
                n_base      = len(base_scores)
            else:
                c = df_cur[df_cur["tag"]  == tag]
                b = df_base[df_base["tag"] == tag]
                cur_scores  = c["judge_score"].values
                base_scores = b["judge_score"].values
                poor_cur    = (c["judge_verdict"] == "poor").mean()
                poor_base   = (b["judge_verdict"] == "poor").mean()
                n_cur, n_base = len(cur_scores), len(base_scores)

            if n_cur < self.min_samples or n_base < self.min_samples:
                continue   # not enough data to be confident

            # ── Test 1: Mann-Whitney U (score distribution) ────────────────
            _, p_mw   = stats.mannwhitneyu(cur_scores, base_scores, alternative="less")
            mean_delta = cur_scores.mean() - base_scores.mean()

            if p_mw < self.p_threshold and mean_delta < -self.delta_warning:
                sev = "critical" if mean_delta < -self.delta_critical else "warning"
                alerts.append(RegressionAlert(
                    kind="quality_drop", tag=label,
                    baseline_mean=float(base_scores.mean()),
                    current_mean =float(cur_scores.mean()),
                    delta=float(mean_delta), p_value=float(p_mw),
                    severity=sev,
                    message=(
                        f"Quality drop [{label}]: "
                        f"{base_scores.mean():.3f} → {cur_scores.mean():.3f} "
                        f"(Δ={mean_delta:+.3f}, p={p_mw:.4f})"
                    ),
                ))

            # ── Test 2: two-proportion z-test (poor rate) ──────────────────
            poor_delta = poor_cur - poor_base
            if poor_delta > 0.05 and n_cur >= self.min_samples:
                p_pool = (poor_cur * n_cur + poor_base * n_base) / (n_cur + n_base)
                se     = (p_pool * (1 - p_pool) * (1/n_cur + 1/n_base)) ** 0.5
                z      = poor_delta / se if se > 0 else 0
                p_z    = 1 - stats.norm.cdf(z)
                if p_z < self.p_threshold:
                    sev = "critical" if poor_delta > 0.10 else "warning"
                    alerts.append(RegressionAlert(
                        kind="poor_rate_spike", tag=label,
                        baseline_mean=float(poor_base),
                        current_mean =float(poor_cur),
                        delta=float(poor_delta), p_value=float(p_z),
                        severity=sev,
                        message=(
                            f"Poor rate spike [{label}]: "
                            f"{poor_base:.1%} → {poor_cur:.1%} "
                            f"(Δ={poor_delta:+.1%}, p={p_z:.4f})"
                        ),
                    ))

        return alerts


# ── Inject degraded samples into last 6h window ──────────────────────────────
now = datetime.utcnow()
print("Injecting simulated quality regression into last 6h window...\n")
degraded_samples = []
for _ in range(60):
    tag = random.choice(["qa", "qa", "code"])
    s   = make_fake_sample(tag)
    s.ts            = now - timedelta(hours=random.uniform(0.0, 5.9))
    s.judge_score   = random.uniform(0.28, 0.52)   # significantly worse
    s.judge_verdict = "poor" if s.judge_score < 0.45 else "ok"
    degraded_samples.append(s)
store.write(degraded_samples)

# ── Run detector ─────────────────────────────────────────────────────────────
detector = RegressionDetector(baseline_hours=72, current_hours=6, min_samples=20)
alerts   = detector.detect(store, now=now)

if alerts:
    print(f"🚨 {len(alerts)} regression alert(s) fired:\n")
    for a in alerts:
        icon = "🔴" if a.severity == "critical" else "🟡"
        print(f"  {icon} [{a.severity.upper():8s}] {a.message}")
else:
    print("✅ No regressions detected")

## Part 6: OTel integration — eval quality as Prometheus metrics

In L48 you traced LLM calls with OTel **spans** ("what happened for this specific request"). Now you add OTel **metric instruments** — aggregated counters and histograms that flow into Prometheus and Grafana.

```
OTel signal types:
  Traces  → per-request span tree (L48 lesson)
  Metrics → aggregated time-series (THIS section)
  Logs    → structured text events
```

Key metric instruments for eval:

| Instrument | Name | What it measures |
|-----------|------|------------------|
| `Counter` | `eval.judged.total` | Total samples judged (ever increasing) |
| `Counter` | `eval.poor.total` | Total poor-verdict samples |
| `Histogram` | `eval.judge.score` | Score distribution (→ percentiles in Prometheus) |
| `UpDownCounter` | `eval.queue.depth` | Current eval queue backlog |

In Prometheus, `eval_judge_score_bucket` becomes queryable:
```promql
# Alert: mean score < 0.70 for 30 minutes
rate(eval_judge_score_sum[1h]) / rate(eval_judge_score_count[1h]) < 0.70

# Alert: poor rate > 20%
rate(eval_poor_total[1h]) / rate(eval_judged_total[1h]) > 0.20
```

In [ ]:
# Cell 15: OTel metric instruments + quality dashboard

# ── Setup OTel MeterProvider with console exporter (Colab-friendly) ──────────
reader   = PeriodicExportingMetricReader(
    ConsoleMetricExporter(),
    export_interval_millis=60_000,   # export every 60s in demo; use 15s in prod
)
provider = MeterProvider(metric_readers=[reader])
metrics.set_meter_provider(provider)
meter    = metrics.get_meter("eval.quality", version="1.0.0")

# ── Define instruments ────────────────────────────────────────────────────────
judged_counter = meter.create_counter(
    "eval.judged.total",
    description="Total samples sent for eval judging",
    unit="1",
)
poor_counter = meter.create_counter(
    "eval.poor.total",
    description="Total samples with poor quality verdict",
    unit="1",
)
score_histogram = meter.create_histogram(
    "eval.judge.score",
    description="Distribution of LLM judge quality scores (0–1)",
    unit="1",
)
queue_gauge = meter.create_up_down_counter(
    "eval.queue.depth",
    description="Current number of samples waiting for eval judging",
    unit="1",
)


def record_eval_result(sample: EvalSample):
    """Record one judged sample into OTel instruments. Call after judging."""
    attrs = {"model": sample.model, "tag": sample.tag}
    judged_counter.add(1, attrs)
    if sample.judge_score is not None:
        score_histogram.record(sample.judge_score, attrs)
    if sample.judge_verdict == "poor":
        poor_counter.add(1, attrs)


# Record all samples from synthetic history
print("Recording eval results into OTel metric instruments...")
for s in all_samples[:200]:  # record 200 for demo
    record_eval_result(s)
print(f"  Recorded 200 samples into counters + histogram")

print("""
What gets exported to Prometheus (every 15s in production):

  # HELP eval_judged_total Total samples sent for eval judging
  eval_judged_total{model="claude-haiku-4-5",tag="qa"}     1234
  eval_judged_total{model="claude-haiku-4-5",tag="code"}    456

  # HELP eval_judge_score Distribution of LLM judge quality scores
  eval_judge_score_bucket{tag="qa",le="0.5"}   89
  eval_judge_score_bucket{tag="qa",le="0.75"}  678
  eval_judge_score_bucket{tag="qa",le="1.0"}   1234
  eval_judge_score_sum{tag="qa"}               987.3
  eval_judge_score_count{tag="qa"}             1234

Grafana alert rule (YAML):
  - alert: EvalQualityDegradation
    expr: |
      rate(eval_judge_score_sum[1h])
      / rate(eval_judge_score_count[1h]) < 0.70
    for: 30m
    labels: {severity: critical}
    annotations:
      summary: "Mean eval score below 0.70 for 30m"
      runbook: "https://wiki.internal/runbooks/eval-quality-degradation"
""")

# ── Quality Dashboard ─────────────────────────────────────────────────────────
df_quality = store.rolling_quality()
df_quality["hour"] = pd.to_datetime(df_quality["hour"])

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
fig.suptitle("Eval Quality Dashboard — 7-Day View", fontsize=14, fontweight="bold")

COLORS = {"qa": "#4C72B0", "summarize": "#DD8452", "search": "#55A868", "code": "#C44E52"}

# Plot 1: mean score by tag over time
for tag in ["qa", "summarize", "search", "code"]:
    sub = df_quality[df_quality["tag"] == tag].sort_values("hour")
    axes[0].plot(sub["hour"], sub["mean_score"],
                 color=COLORS[tag], marker=".", linewidth=1.5, label=tag)

regression_start = pd.Timestamp(now - timedelta(hours=6))
axes[0].axvline(x=regression_start, color="red", linestyle="--", alpha=0.7, label="regression window")
axes[0].axhline(y=0.70, color="orange", linestyle=":", alpha=0.8, linewidth=2, label="warning (0.70)")
axes[0].set_ylabel("Mean Judge Score", fontsize=11)
axes[0].set_title("Quality Score Over Time (per tag)", fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0.3, 1.0)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %Hh"))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30)

# Plot 2: poor rate by tag
for tag in ["qa", "summarize", "search", "code"]:
    sub = df_quality[df_quality["tag"] == tag].sort_values("hour")
    axes[1].plot(sub["hour"], sub["poor_rate"] * 100,
                 color=COLORS[tag], marker=".", linewidth=1.5, label=tag)

axes[1].axvline(x=regression_start, color="red", linestyle="--", alpha=0.7, label="regression window")
axes[1].axhline(y=20, color="orange", linestyle=":", alpha=0.8, linewidth=2, label="warning (20%)")
axes[1].set_ylabel("Poor Verdict Rate (%)", fontsize=11)
axes[1].set_xlabel("Time (UTC)", fontsize=11)
axes[1].set_title("Poor Verdict Rate Over Time (per tag)", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %Hh"))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig("eval_quality_dashboard.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ Dashboard saved to eval_quality_dashboard.png")
print("   Note the quality drop in the last 6h (regression window, red dashed line)")

## 10 Pitfalls in Eval at Scale

| # | Pitfall | Consequence | Fix |
|---|---------|-------------|-----|
| 1 | **Judging on the hot path** | Adds judge latency (0.3–1s) to user P99 | Always decouple via async queue |
| 2 | **Random-only sampling** | Misses rare but impactful failure modes | Add importance sampling for low-confidence, errors |
| 3 | **Judge model version drift** | Scores not comparable across time if judge model changes | Store `judge_model_version` in metric table; never silently upgrade judge |
| 4 | **Alerting on tiny windows** | 5 bad samples in 1 hour → false alarm → alert fatigue | Require `min_samples >= 30` before any statistical test |
| 5 | **Threshold-only regression** | Misses gradual 2-week drift that never crosses threshold | Use Mann-Whitney U to test distribution shift across windows |
| 6 | **Not tagging samples by query type** | Can't tell if regression is in code gen vs summarization | Always store `tag`, `model`, `version` in metric row |
| 7 | **Only eval on golden set** | Production traffic has very different distribution than golden set | Sample real production traffic; supplement, don't replace, with golden set |
| 8 | **Trusting judge score as ground truth** | Judge can be wrong, especially for domain-specific tasks | Calibrate judge against human labels monthly; track judge accuracy |
| 9 | **Not sampling errors at 100%** | Error cases (exceptions, refusals, timeouts) masked in mean score | Always enqueue errors regardless of sampling rate |
| 10 | **Storing only aggregate metrics** | Can't root-cause: which queries degraded, what did they look like? | Store raw `(id, query, response, score)` — aggregates are always recomputable |

In [ ]:
# Cell 17: Full integrated ProductionEvalSystem + summary

class ProductionEvalSystem:
    """
    Top-level class wiring sampler + pipeline + batch judge +
    metric store + regression detector + OTel instruments.

    Deployment pattern:
        system = ProductionEvalSystem()
        # In inference handler (hot path):
        system.on_inference_complete(sample)    # non-blocking, 1μs
        # Scheduled job (every hour):
        await system.run_eval_worker()          # drains queue, judges, stores
        alerts = system.check_regressions()     # statistical tests
    """

    def __init__(
        self,
        sample_rate: float = 0.05,
        batch_size:  int   = 50,
        db_path:     str   = ":memory:",
    ):
        self.sampler  = EvalSampler(
            strategy   = SampleStrategy.IMPORTANCE,
            base_rate  = sample_rate,
            tag_rates  = {"code": sample_rate * 3},
        )
        self.pipeline = AsyncEvalPipeline(sampler=self.sampler, batch_size=batch_size)
        self.judge    = BatchJudge(model=HAIKU)
        self.store    = EvalMetricStore(db_path=db_path)
        self.detector = RegressionDetector(baseline_hours=48, current_hours=6)
        self._n_requests = 0

    def on_inference_complete(self, sample: EvalSample):
        """Call after every production inference. Non-blocking."""
        self._n_requests += 1
        enqueued = self.pipeline.maybe_enqueue(sample)
        if enqueued:
            queue_gauge.add(1)

    async def run_eval_worker(self, n_batches: int = 3):
        """Background job: drain queue, judge, write to store + OTel."""
        await self.pipeline._worker(max_batches=n_batches)
        newly_judged = self.pipeline._judged
        if newly_judged:
            n = self.store.write(newly_judged)
            for s in newly_judged:
                record_eval_result(s)
                queue_gauge.add(-1)
            print(f"  Stored {n} scored samples")

    def check_regressions(self) -> list[RegressionAlert]:
        """Run regression detection. Call on a schedule (e.g., every hour)."""
        return self.detector.detect(self.store)

    def status(self) -> dict:
        return {
            "total_requests":  self._n_requests,
            "sampled_for_eval": self.pipeline._enqueued,
            "actual_rate":     round(self.pipeline._enqueued / max(1, self._n_requests), 4),
            "queue_depth":     self.pipeline._queue.qsize(),
            "dropped":         self.pipeline._dropped,
        }


# ── Full system demo ──────────────────────────────────────────────────────────
print("=" * 60)
print("PRODUCTION EVAL SYSTEM — INTEGRATION DEMO")
print("=" * 60)

system = ProductionEvalSystem(sample_rate=0.20, batch_size=8)

# Simulate 150 production inference requests (hot path)
print("\nSimulating 150 production inference requests (hot path)...", end="")
for i in range(150):
    tag = random.choice(["qa", "qa", "summarize", "search", "code"])
    s   = make_fake_sample(tag, confidence=0.45 if tag == "code" else 0.85)
    s.query    = f"User request #{i}"
    s.response = "Answer. " * random.randint(8, 60)
    system.on_inference_complete(s)
print(" done")

print("\nSystem status (inference complete, eval pending):")
for k, v in system.status().items():
    print(f"  {k:20s}: {v}")

print("\nRunning background eval worker...")
asyncio.run(system.run_eval_worker(n_batches=3))

print("\nRunning regression detection...")
system.store.write(degraded_samples)  # inject same degraded data
regression_alerts = system.check_regressions()
if regression_alerts:
    for a in regression_alerts:
        icon = "🔴" if a.severity == "critical" else "🟡"
        print(f"  {icon} {a.message}")
else:
    print("  ✅ No regressions")

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("LESSON 49 SUMMARY")
print("=" * 60)
summary_rows = [
    ("Sampling",    "Importance > Stratified > Random. Always 100% sample errors."),
    ("Decoupling",  "asyncio.Queue between inference and eval — never blocks users."),
    ("Batch judge", "Message Batches API: 50% cheaper, poll until 'ended'."),
    ("Metric store","SQLite (dev) → TimescaleDB (prod). Store raw samples, not just means."),
    ("Regression",  "Mann-Whitney U for score drift; z-test for poor-rate spike."),
    ("OTel",        "Counter + Histogram → Prometheus → Grafana alerts."),
    ("The rule",    "≥100 judged samples/day/metric to get meaningful CI."),
]
df_sum = pd.DataFrame(summary_rows, columns=["Concept", "Key Insight"])
print(df_sum.to_string(index=False))

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
HOMEWORK (5 tasks)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. CUSUM drift detector: implement Page-Hinkley CUSUM for
   detecting gradual quality decay. Compare sensitivity to
   Mann-Whitney U on a slow-drift scenario (−0.5% per day
   over 30 days). Which detects it earlier?

2. Judge calibration checker: build JudgeCalibration with 20
   human-labeled samples. Compute Cohen's kappa between judge
   verdicts and human verdicts. Alert when kappa < 0.6.

3. Error-first sampling: modify maybe_enqueue() to always
   enqueue when latency_s > SLO or judge_score < 0.3,
   regardless of base_rate. Verify 100% capture of bad cases.

4. Reservoir validation: implement add_to_reservoir() fully
   and verify it is unbiased — compare score distribution of
   reservoir sample vs full population using KS test.

5. Wire into AutoResearcher: add ProductionEvalSystem to the
   L36 multi-agent swarm. Track quality per-agent (searcher /
   critic / synthesizer) using tag = agent_name. Which agent
   degrades first under load?

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NEXT → Lesson 50: Track 5 Capstone — AgentOps in Production
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
L50 wires ALL of Track 5 into one deployable production stack
on top of the L36 Multi-Agent Research Swarm:

  L46 Durable Execution  →  crash-safe pipeline checkpointing
  L47 GPU Autoscaling    →  vLLM HPA with queue-depth signal
  L48 OTel Tracing       →  per-span breakdown of every call
  L49 Eval at Scale      →  continuous quality monitoring

Final deliverable: docker-compose with Grafana dashboard
showing traces + metrics + eval scores side-by-side.
Open-source contribution target: auto_researcher/agentops/ 🚀
""")